# 02 — Population Estimation and Panel Construction

This notebook:

1. Standardizes LGU names across the FOI dengue data and PSA reference files.
2. Estimates population for 2021–2023 (interpolation) and 2025 (extrapolation) from the 2020 and 2024 PSA Census anchors and uses the 2024 figure directly.
3. Recomputes population density for every LGU-year.
4. Merges case counts with population/density into the final 85-row LGU-year panel.
5. Runs calculation validation round 1 - hand-recomputing a sample of values independently of the pipeline code.

All transformation logic lives in `src/population.py` and `src/panel.py`, and is covered by `tests/test_population.py` and `tests/test_panel.py` (`pytest -v`).

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.lgu_names import normalize_lgu_name
from src.population import build_population_panel, estimate_population, ALL_YEARS
from src.panel import build_lgu_year_panel

RAW = Path.cwd().parent / "data" / "01_raw"
REF = Path.cwd().parent / "data" / "02_official_reference"
PROCESSED = Path.cwd().parent / "data" / "03_processed"
VALIDATED = Path.cwd().parent / "data" / "04_validated"

PROCESSED.mkdir(exist_ok=True)
VALIDATED.mkdir(exist_ok=True)

## 1. LGU name standardization

Both source files already use a consistent short-form naming convention, but names are passed through `normalize_lgu_name()` regardless to make the pipeline robust to future source files that use the PSA official form or stray whitespace/case,and fails loudly (`ValueError`) on any name outside the 17 NCR LGUs.

In [2]:
cases_wide = pd.read_csv(RAW / "dengue_datasets__local_govt_level.csv", thousands=",")
pop_2020 = pd.read_csv(REF / "ncr_population_masterlist__2020.csv")
pop_2024 = pd.read_csv(REF / "ncr_population_masterlist__2024.csv")

# Confirm every name in every source resolves to one of the 17 LGUs
for label, df, col in [
    ("FOI dengue cases", cases_wide, "LGU"),
    ("PSA 2020", pop_2020, "City and Municipality"),
    ("PSA 2024", pop_2024, "City and Municipality"),
]:
    normalized = df[col].map(normalize_lgu_name)
    print(f"{label}: {normalized.nunique()} unique LGUs, all resolved OK")

FOI dengue cases: 17 unique LGUs, all resolved OK
PSA 2020: 17 unique LGUs, all resolved OK
PSA 2024: 17 unique LGUs, all resolved OK


## 2. Population interpolation (2021–2023) and extrapolation (2025)

Two official anchors are used: the 2020 PSA Census of Population and Housing,
and the 2024 PSA Census of Population (POPCEN). A single linear formula
drives both interpolation and extrapolation:

```
P(t) = P0 + (P1 - P0) * (t - 2020) / (2024 - 2020)
```

- **2021, 2022, 2023** fall *between* the two anchors → **interpolation**,
  bounded by real data on both sides.
- **2024** is read *directly* from the PSA source file — never derived from
  the formula.
- **2025** falls *beyond* the last anchor → **extrapolation**: the same
  formula, continuing the observed 2020–2024 growth rate forward by one year.
  This carries more uncertainty than interpolation, since no second anchor
  constrains it, and is flagged as a lower-confidence estimate throughout the
  dataset and dashboard.

In [3]:
# Worked example: Quezon City, to make the formula concrete before applying it to all 17 LGUs.
qc_p0 = pop_2020.set_index("City and Municipality").loc["Quezon City", "Total Population"]
qc_p1 = pop_2024.set_index("City and Municipality").loc["Quezon City", "Total Population"]

print(f"Quezon City — 2020 anchor: {qc_p0:,}, 2024 anchor: {qc_p1:,}\n")
for year in ALL_YEARS:
    est = estimate_population(qc_p0, qc_p1, year)
    kind = "official" if year == 2024 else ("interpolated" if year < 2024 else "extrapolated")
    print(f"  {year} ({kind:>12}): {est:,.2f}")

Quezon City — 2020 anchor: 2,960,048, 2024 anchor: 3,084,270

  2021 (interpolated): 2,991,103.50
  2022 (interpolated): 3,022,159.00
  2023 (interpolated): 3,053,214.50
  2024 (    official): 3,084,270.00
  2025 (extrapolated): 3,115,325.50


In [4]:
population_panel = build_population_panel(pop_2020, pop_2024)

print(f"Population panel shape: {population_panel.shape}")
population_panel.groupby("Status").size()

Population panel shape: (85, 6)


Status
extrapolated    17
interpolated    51
official        17
dtype: int64

## 3. Unit test: population estimation against the 2020/2024 anchors

The full set of assertions lives in `tests/test_population.py` (19 tests) —
run via `pytest -v tests/test_population.py`. The checks confirm:

- `estimate_population()` at 2020 and 2024 returns the anchor value unchanged
  (no drift from the formula at the boundary).
- 2021–2023 estimates fall strictly between the two anchors and are evenly
  spaced (true linear interpolation).
- The 2025 estimate continues the same per-year growth increment observed
  between 2020 and 2024 (true linear extrapolation, same formula).
- A declining LGU (e.g. Makati, whose 2024 figure is lower than its 2020
  figure) extrapolates *further down* for 2025, not back upward, and confirms
  the formula responds correctly to the direction of the underlying trend.
- Years before 2020 are rejected since the study's panel starts at 2021.
- Every LGU's 2024 panel value matches the official PSA source exactly (
  it was read directly, not recomputed by the interpolation formula).

In [5]:
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "-v", "tests/test_population.py"],
    cwd=Path.cwd().parent, capture_output=True, text=True,
)
print(result.stdout[-2500:])

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- c:\Users\USER\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\USER\capstone2_dengue_ncr
configfile: pytest.ini
plugins: anyio-4.7.0
collecting ... collected 19 items

tests/test_population.py::test_anchors_are_2020_and_2024 PASSED          [  5%]
tests/test_population.py::test_estimate_at_2020_returns_p0_unchanged PASSED [ 10%]
tests/test_population.py::test_estimate_at_2024_returns_p1_unchanged PASSED [ 15%]
tests/test_population.py::test_interpolated_years_fall_strictly_between_anchors_for_growth PASSED [ 21%]
tests/test_population.py::test_interpolation_is_linear_and_evenly_spaced PASSED [ 26%]
tests/test_population.py::test_extrapolation_continues_the_same_growth_rate_beyond_2024 PASSED [ 31%]
tests/test_population.py::test_extrapolation_uses_same_formula_as_interpolation PASSED [ 36%]
tests/test_population.py::test_declining_

## 4. Population density recomputation

Density is recomputed per LGU-year as `Population ÷ Land Area`, using the
year-specific population estimate. Land area is fixed and verified identical
between the 2020 and 2024 PSA source files, so it is carried forward unchanged
across all five years for a given LGU.

In [6]:
population_panel[population_panel["LGU"] == "Quezon City"]

,LGU,Year,Population,Land Area,Population Density,Status
65,Quezon City,2021,2991103.5,171.71,17419.506726,interpolated
66,Quezon City,2022,3022159.0,171.71,17600.366898,interpolated
67,Quezon City,2023,3053214.5,171.71,17781.227069,interpolated
68,Quezon City,2024,3084270.0,171.71,17962.087240,official
69,Quezon City,2025,3115325.5,171.71,18142.947411,extrapolated


## 5. Merge into the 85-row LGU-year panel

The FOI case-count table (wide format: one row per LGU, one column per year)
is reshaped to long format and merged against the population/density panel on
`(LGU, Year)`, using a validated one-to-one merge so a silent row drop or
duplication fails loudly rather than shrinking or inflating the panel.

In [7]:
lgu_year_panel = build_lgu_year_panel(cases_wide, population_panel)

print(f"Final panel shape: {lgu_year_panel.shape}")
assert lgu_year_panel.shape[0] == 85, "Panel must have exactly 85 rows (17 LGUs x 5 years)"
lgu_year_panel.head(10)

Final panel shape: (85, 8)


,LGU,Year,Dengue Cases,Population,Land Area,Population Density,Incidence Rate,Status
0,Caloocan,2021,1051,1674424.25,55.80,30007.603047,62.767844,interpolated
1,Caloocan,2022,4371,1687264.50,55.80,30237.715054,259.058375,interpolated
2,Caloocan,2023,2334,1700104.75,55.80,30467.827061,137.285658,interpolated
3,Caloocan,2024,3262,1712945.00,55.80,30697.939068,190.432267,official
4,Caloocan,2025,5451,1725785.25,55.80,30928.051075,315.856217,extrapolated
5,Las Piñas,2021,495,608607.00,32.69,18617.528296,81.333274,interpolated
6,Las Piñas,2022,1603,610921.00,32.69,18688.314469,262.390718,interpolated
7,Las Piñas,2023,969,613235.00,32.69,18759.100642,158.014464,interpolated
8,Las Piñas,2024,1506,615549.00,32.69,18829.886816,244.659645,official
9,Las Piñas,2025,2132,617863.00,32.69,18900.672989,345.060313,extrapolated


## 6. Unit test: merge returns exactly 85 rows

Covered in `tests/test_panel.py` - it confirms that the melt step alone produces 85
rows, the merge preserves all 85 after joining with the population panel, no
duplicate `(LGU, Year)` keys are introduced, no nulls are produced, and a
deliberately truncated population panel (missing one LGU-year) causes the
merge to raise rather than silently drop a row.

In [8]:
result = subprocess.run(
    ["python", "-m", "pytest", "-v", "tests/test_panel.py"],
    cwd=Path.cwd().parent, capture_output=True, text=True,
)
print(result.stdout[-2500:])

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- c:\Users\USER\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\USER\capstone2_dengue_ncr
configfile: pytest.ini
plugins: anyio-4.7.0
collecting ... collected 10 items

tests/test_panel.py::test_melt_produces_85_rows PASSED                   [ 10%]
tests/test_panel.py::test_melt_preserves_a_known_value PASSED            [ 20%]
tests/test_panel.py::test_merged_panel_returns_exactly_85_rows PASSED    [ 30%]
tests/test_panel.py::test_merged_panel_has_17_lgus_and_5_years PASSED    [ 40%]
tests/test_panel.py::test_merged_panel_has_no_duplicate_keys PASSED      [ 50%]
tests/test_panel.py::test_merged_panel_has_no_null_values PASSED         [ 60%]
tests/test_panel.py::test_merge_raises_if_a_lgu_year_is_missing_from_population PASSED [ 70%]
tests/test_panel.py::test_quezon_city_2025_row_values PASSED             [ 80%]
tests/test_panel.py::

## 7. Calculation validation round 1 — hand-recompute a sample

Five LGU-years are hand-recomputed from the raw source CSVs using plain
arithmetic, independent of `src/population.py` and `src/panel.py`, and
compared against the pipeline's output. The sample spans all three population
status types (interpolated, official, extrapolated) and a range of LGU sizes.

Full write-up: `docs/calculation_validation_round1.md`.

In [9]:
pop_2020_idx = pop_2020.set_index("City and Municipality")
pop_2024_idx = pop_2024.set_index("City and Municipality")
cases_idx = cases_wide.set_index("LGU")

samples = [
    ("Quezon City", 2021, "interpolated"),
    ("Manila", 2024, "official"),
    ("Pateros", 2025, "extrapolated"),
    ("Taguig", 2023, "interpolated"),
    ("Malabon", 2025, "extrapolated"),
]

rows = []
for lgu, year, status in samples:
    p0 = pop_2020_idx.loc[lgu, "Total Population"]
    p1 = pop_2024_idx.loc[lgu, "Total Population"]
    land = pop_2020_idx.loc[lgu, "Land Area"]

    hand_pop = p0 if year == 2020 else p1 if year == 2024 else p0 + (p1 - p0) * (year - 2020) / (2024 - 2020)
    hand_density = hand_pop / land
    hand_ir = cases_idx.loc[lgu, str(year)] / hand_pop * 100_000

    pipe_row = lgu_year_panel[(lgu_year_panel["LGU"] == lgu) & (lgu_year_panel["Year"] == year)].iloc[0]

    rows.append({
        "LGU": lgu, "Year": year, "Status": status,
        "Pop (hand)": round(hand_pop, 2), "Pop (pipeline)": round(pipe_row["Population"], 2),
        "Density (hand)": round(hand_density, 2), "Density (pipeline)": round(pipe_row["Population Density"], 2),
        "IR (hand)": round(hand_ir, 2), "IR (pipeline)": round(pipe_row["Incidence Rate"], 2),
    })

validation_df = pd.DataFrame(rows)
validation_df["Pop match"] = (validation_df["Pop (hand)"] - validation_df["Pop (pipeline)"]).abs() < 0.01
validation_df["Density match"] = (validation_df["Density (hand)"] - validation_df["Density (pipeline)"]).abs() < 0.01
validation_df["IR match"] = (validation_df["IR (hand)"] - validation_df["IR (pipeline)"]).abs() < 0.01

assert validation_df[["Pop match", "Density match", "IR match"]].all().all(), "Calculation validation round 1 FAILED"
print("Calculation validation round 1: all 5 sampled LGU-years match hand-computed values.\n")
validation_df

Calculation validation round 1: all 5 sampled LGU-years match hand-computed values.



,LGU,Year,Status,Pop (hand),Pop (pipeline),Density (hand),Density (pipeline),IR (hand),IR (pipeline),Pop match,Density match,IR match
0,Quezon City,2021,interpolated,2991103.50,2991103.50,17419.51,17419.51,55.06,55.06,True,True,True
1,Manila,2024,official,1902590.00,1902590.00,76164.53,76164.53,289.66,289.66,True,True,True
2,Pateros,2025,extrapolated,67842.00,67842.00,6523.27,6523.27,347.87,347.87,True,True,True
3,Taguig,2023,interpolated,1202744.25,1202744.25,26603.50,26603.50,222.99,222.99,True,True,True
4,Malabon,2025,extrapolated,392280.75,392280.75,24970.13,24970.13,343.89,343.89,True,True,True


## 8. Write outputs

- `data/03_processed/population_panel.csv` — long-format population/density,
  17 LGUs × 5 years, with status tags.
- `data/04_validated/lgu_year_panel.csv` — final merged 85-row analytical
  panel, ready for the descriptive analysis and regression steps in
  notebook 03.

In [10]:
population_panel.to_csv(PROCESSED / "population_panel.csv", index=False)
lgu_year_panel.to_csv(VALIDATED / "lgu_year_panel.csv", index=False)

print("Written:")
print(f"  {PROCESSED / 'population_panel.csv'}  {population_panel.shape}")
print(f"  {VALIDATED / 'lgu_year_panel.csv'}  {lgu_year_panel.shape}")

Written:
  c:\Users\USER\capstone2_dengue_ncr\data\03_processed\population_panel.csv  (85, 6)
  c:\Users\USER\capstone2_dengue_ncr\data\04_validated\lgu_year_panel.csv  (85, 8)


## Summary

- Population estimated for all 85 LGU-years: official (2020, 2024),
  interpolated (2021–2023), extrapolated (2025 — flagged lower-confidence).
- Density recomputed per LGU-year from the corresponding population estimate.
- Case counts merged into a validated 85-row LGU-year panel — no missing or
  duplicated rows.
- Data dictionary: `docs/data_dictionary.md`.
- Calculation validation round 1: `docs/calculation_validation_round1.md` — no
  discrepancies found.

**Next:** notebook 03 covers the descriptive indicators (incidence,
year-over-year change, and each LGU's five-year average) and the NCR-level
age-sex profile. Notebook 04 fits the Poisson baseline and measures the
overdispersion that motivates the Negative Binomial specification.